# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading, exploring, and analyzing the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # Metadata is an object, not a dict
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, their fields, and associated `@id` values.

In [ ]:
# List available record sets, their @ids, and fields
print("Available record sets and their fields:")
for rs in dataset.record_sets:
    print(f"- RecordSet @id: {rs.id}, name: {rs.name}")
    for field in rs.fields:
        print(f"    - Field @id: {field.id}, name: {field.name}, dataType: {field.data_type}")

In [ ]:
# Preview a few records from each record set
for rs in dataset.record_sets:
    print(f"\nRecords sample for RecordSet '{rs.name}' (@id={rs.id}):")
    records = dataset.records(record_set=rs.id)
    for i, rec in enumerate(records):
        print(json.dumps(rec, indent=2))
        if i >= 1:
            break

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the RecordSet and Field `@id`s from above.

In [ ]:
# Collect all RecordSet @id values
record_set_ids = [rs.id for rs in dataset.record_sets]
print(f"RecordSet IDs detected: {record_set_ids}")

# Load all record sets into individual DataFrames
dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records for RecordSet {record_set_id}")

# For demonstration, use the first tabular record set
main_record_set_id = None
for rs in dataset.record_sets:
    if rs.id in dataframes:  # At least some records exist
        main_record_set_id = rs.id
        main_record_set_name = rs.name
        print(f"Using RecordSet '{main_record_set_name}' with @id: {main_record_set_id}")
        print(f"Fields: {[f'@id: {field.id}, name: {field.name}' for field in rs.fields]}")
        break

# Show columns for that RecordSet
if main_record_set_id:
    print(dataframes[main_record_set_id].columns.tolist())
    dataframes[main_record_set_id].head()
else:
    print("No tabular record set found.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering, normalization, or grouping. We'll select a numeric field, remove low values, normalize it, and group by a category where appropriate.

> All field names are referenced by their `@id`s as required.

In [ ]:
# Pick a numeric field for analysis based on the main RecordSet fields
main_rs = None
for rs in dataset.record_sets:
    if rs.id == main_record_set_id:
        main_rs = rs
        break

# Identify a numeric field candidate
numeric_field_id = None
for field in main_rs.fields:
    # Choose the first field with type Float or Integer
    if field.data_type in ["Float", "Integer", "Number"]:
        numeric_field_id = field.id
        print(f"Using numeric field: {field.name} (@id={numeric_field_id})")
        break

df = dataframes[main_record_set_id]
if numeric_field_id is None or numeric_field_id not in df.columns:
    print("No numeric field found. Please check available fields.")
else:
    # Attempt to convert to float for EDA
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    threshold = df[numeric_field_id].mean() if not df[numeric_field_id].isnull().all() else 0
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records where {numeric_field_id} > {threshold:.2f} (mean): {len(filtered_df)} rows")
    # Normalization
    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    )
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
    # Try to find a groupable/categorical field
    group_field_id = None
    for field in main_rs.fields:
        # Common group fields: look for type 'Text' or multiple values, e.g. 'Sample name', 'Condition', etc.
        if field.data_type == "Text" and field.id != numeric_field_id:
            group_field_id = field.id
            print(f"Grouping by field: {field.name} (@id={group_field_id})")
            break
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(grouped_df.head())

## 5. Visualization
Visualize numeric distributions and/or compare group averages.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id and numeric_field_id in df.columns:
    plt.figure(figsize=(6,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f'Distribution for field {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.show()
    
    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(8,5))
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
        plt.title(f'{numeric_field_id} by {group_field_id}')
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

## 6. Conclusion
In this notebook, we loaded the FAIR^2 dataset's Croissant schema, inspected its record sets and fields using their `@id`s, extracted tabular data, performed filtering and normalization on a numeric field, grouped by a categorical field, and visualized distributions. The use of `@id` throughout allows for robust, schema-driven exploration.

Further analysis could include deeper cross-sample comparisons, integration with UniProt for additional protein annotation, or more sophisticated statistical modeling.